## 创建Mesh index医学词典

In [6]:
import re
from collections import defaultdict
from lxml import etree
import json
from pathlib import Path
from itertools import islice

In [3]:
def normalize_term(term: str) -> str:
    term = str(term).strip().lower()
    term = term.replace("–", "-").replace("—", "-")
    term = re.sub(r"\s+", " ", term)
    return term


def parse_mesh_descriptors(xml_path):
    """
    流式解析MeSH Descriptor XML，避免一次加载整个XML。
    """
    concepts = []
    alias_index = defaultdict(list)

    context = etree.iterparse(
        str(xml_path),
        events=("end",),
        tag="DescriptorRecord"
    )

    for _, record in context:
        mesh_id = record.findtext("DescriptorUI", "").strip()

        preferred_term = record.findtext(
            "DescriptorName/String",
            ""
        ).strip()

        terms = {
            node.text.strip()
            for node in record.findall(".//Term/String")
            if node.text and node.text.strip()
        }

        if preferred_term:
            terms.add(preferred_term)

        tree_numbers = [
            node.text.strip()
            for node in record.findall(
                "TreeNumberList/TreeNumber"
            )
            if node.text
        ]

        concept = {
            "concept_id": mesh_id,
            "preferred_term": preferred_term,
            "synonyms": sorted(terms),
            "source": "MeSH",
            "tree_numbers": tree_numbers
        }

        concepts.append(concept)

        for term in terms:
            normalized = normalize_term(term)

            if mesh_id not in alias_index[normalized]:
                alias_index[normalized].append(mesh_id)

        # 释放已经解析的XML节点
        record.clear()

        while record.getprevious() is not None:
            del record.getparent()[0]

    return concepts, dict(alias_index)

In [4]:
mesh_concepts, mesh_alias_index = (
    parse_mesh_descriptors(
        r"F:\RAG\data\desc2026.xml"
    )
)

print("Concepts:", len(mesh_concepts))
print(
    mesh_alias_index.get(
        "myocardial infarction"
    )
)

Concepts: 31110
['D009203']


In [5]:
output_dir = Path(
    r"F:RAG\MedicalTerminology\mesh"
)
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

with (
    output_dir / "mesh_concepts.jsonl"
).open("w", encoding="utf-8") as file:
    for concept in mesh_concepts:
        file.write(
            json.dumps(
                concept,
                ensure_ascii=False
            )
            + "\n"
        )

with (
    output_dir / "mesh_alias_index.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        mesh_alias_index,
        file,
        ensure_ascii=False
    )

In [7]:
alias_path = r"F:\RAG\MedicalTerminology\mesh\mesh_alias_index.json"
concept_path = r"F:\RAG\MedicalTerminology\mesh\mesh_concepts.jsonl"

with open(alias_path, "r", encoding="utf-8") as f:
    alias_index = json.load(f)

print("Alias index type:", type(alias_index))
print("Alias samples:", list(alias_index.items())[:3])

with open(concept_path, "r", encoding="utf-8") as f:
    for line in islice(f, 3):
        print(json.loads(line))

Alias index type: <class 'dict'>
Alias samples: [('a23187, antibiotic', ['D000001']), ('a-23187', ['D000001']), ('antibiotic a23187', ['D000001'])]
{'concept_id': 'D000001', 'preferred_term': 'Calcimycin', 'synonyms': ['4-Benzoxazolecarboxylic acid, 5-(methylamino)-2-((3,9,11-trimethyl-8-(1-methyl-2-oxo-2-(1H-pyrrol-2-yl)ethyl)-1,7-dioxaspiro(5.5)undec-2-yl)methyl)-, (6S-(6alpha(2S*,3S*),8beta(R*),9beta,11alpha))-', 'A 23187', 'A-23187', 'A23187', 'A23187, Antibiotic', 'Antibiotic A23187', 'Calcimycin'], 'source': 'MeSH', 'tree_numbers': ['D02.355.291.933.125', 'D02.540.576.625.125', 'D03.633.100.221.173', 'D04.345.241.654.125', 'D04.345.674.625.125']}
{'concept_id': 'D000002', 'preferred_term': 'Temefos', 'synonyms': ['Abate', 'Difos', 'Temefos', 'Temephos'], 'source': 'MeSH', 'tree_numbers': ['D02.705.400.625.800', 'D02.705.539.345.800', 'D02.886.300.692.800']}
{'concept_id': 'D000003', 'preferred_term': 'Abattoirs', 'synonyms': ['Abattoir', 'Abattoirs', 'House, Slaughter', 'Houses, 